In [ ]:
# Cell 1: Install dependencies
!pip install mtcnn opencv-python-headless scikit-learn -q


In [ ]:
# Cell 2: Imports
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt

import tensorflow as tf
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.applications.efficientnet import preprocess_input
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns

print("TensorFlow version:", tf.__version__)
print("GPU available:", tf.config.list_physical_devices('GPU'))


In [ ]:
# Cell 3: Dataset path & quick sanity check
# ✅ Update this path to your MTCNN-extracted face dataset
DATASET_PATH = "/kaggle/input/datasets/kshitizbhargava/deepfake-face-images/Final Dataset"

# Verify folder structure: expects DATASET_PATH/Fake/ and DATASET_PATH/Real/
classes = os.listdir(DATASET_PATH)
print("Classes found:", classes)

for cls in classes:
    cls_path = os.path.join(DATASET_PATH, cls)
    if os.path.isdir(cls_path):
        count = len(os.listdir(cls_path))
        print(f"  {cls}: {count} images")


In [ ]:
# Cell 4: Separate train and validation data generators
#   - Train generator: WITH augmentation
#   - Val generator:   NO augmentation (critical fix!)

IMG_SIZE   = (224, 224)
BATCH_SIZE = 32

train_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    validation_split=0.2,
    rotation_range=15,
    zoom_range=0.1,
    horizontal_flip=True,
    brightness_range=[0.8, 1.2],
    shear_range=0.05
)

# Separate datagen with NO augmentation for validation
val_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    validation_split=0.2
)

train_generator = train_datagen.flow_from_directory(
    DATASET_PATH,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='binary',
    subset='training',
    shuffle=True,
    seed=42
)

val_generator = val_datagen.flow_from_directory(
    DATASET_PATH,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='binary',
    subset='validation',
    shuffle=False,   # Keep order for evaluation
    seed=42
)

print("\nClass indices:", train_generator.class_indices)
print(f"Train samples  : {train_generator.samples}")
print(f"Val samples    : {val_generator.samples}")


In [ ]:
# Cell 5: Compute class weights to handle imbalanced data
classes_arr = train_generator.classes
weights = compute_class_weight('balanced', classes=np.unique(classes_arr), y=classes_arr)
class_weights = {0: weights[0], 1: weights[1]}
print("Class weights:", class_weights)


In [ ]:
# Cell 6: Build the model
#   - Stronger head: GAP -> Dense(256) -> Dropout -> Dense(1)
#   - ALL base layers frozen initially

base_model = EfficientNetB0(
    weights='imagenet',
    include_top=False,
    input_shape=(224, 224, 3)
)
base_model.trainable = False   # Freeze everything first

x      = base_model.output
x      = GlobalAveragePooling2D()(x)
x      = Dense(256, activation='relu')(x)
x      = Dropout(0.4)(x)
output = Dense(1, activation='sigmoid')(x)

model = Model(inputs=base_model.input, outputs=output)
model.summary(line_length=80)


In [ ]:
# Cell 7: Callbacks (shared across both training phases)

callbacks = [
    EarlyStopping(
        monitor='val_accuracy',
        patience=5,
        restore_best_weights=True,
        verbose=1
    ),
    ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=3,
        min_lr=1e-8,
        verbose=1
    ),
    ModelCheckpoint(
        'best_deepfake_model.keras',
        monitor='val_accuracy',
        save_best_only=True,
        verbose=1
    )
]


In [ ]:
# Cell 8: Phase 1 — Train only the new head (base fully frozen)
#   Higher LR is safe here since pretrained weights are frozen

model.compile(
    optimizer=Adam(learning_rate=1e-3),
    loss='binary_crossentropy',
    metrics=['accuracy', 'AUC']
)

print("=== Phase 1: Training head only ===")
history_phase1 = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=15,
    class_weight=class_weights,
    callbacks=callbacks
)


In [ ]:
# Cell 9: Phase 2 — Unfreeze last 50 layers, fine-tune with VERY low LR
#   Low LR is critical to avoid destroying pretrained weights

for layer in base_model.layers[-50:]:
    layer.trainable = True

model.compile(
    optimizer=Adam(learning_rate=1e-5),   # ← 100x lower than Phase 1
    loss='binary_crossentropy',
    metrics=['accuracy', 'AUC']
)

print("=== Phase 2: Fine-tuning last 50 layers ===")
print(f"Trainable layers now: {sum(1 for l in model.layers if l.trainable)}")

history_phase2 = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=25,
    class_weight=class_weights,
    callbacks=callbacks
)


In [ ]:
# Cell 10: Plot training curves for both phases

def plot_history(h1, h2, metric):
    p1 = h1.history[metric]
    p2 = h2.history[metric]
    vp1 = h1.history[f'val_{metric}']
    vp2 = h2.history[f'val_{metric}']

    combined_train = p1 + p2
    combined_val   = vp1 + vp2
    split_epoch    = len(p1)
    epochs         = range(1, len(combined_train) + 1)

    plt.plot(epochs, combined_train, label=f'Train {metric}')
    plt.plot(epochs, combined_val,   label=f'Val {metric}')
    plt.axvline(x=split_epoch, color='gray', linestyle='--', label='Phase 2 start')
    plt.xlabel('Epoch'); plt.ylabel(metric.capitalize())
    plt.legend(); plt.title(f'{metric.capitalize()} over epochs')

plt.figure(figsize=(14, 5))
plt.subplot(1, 2, 1); plot_history(history_phase1, history_phase2, 'accuracy')
plt.subplot(1, 2, 2); plot_history(history_phase1, history_phase2, 'loss')
plt.tight_layout()
plt.savefig('training_curves.png', dpi=150)
plt.show()


In [ ]:
# Cell 11: Evaluate on validation set

val_generator.reset()
loss, acc, auc = model.evaluate(val_generator, verbose=1)
print(f"\nValidation Accuracy : {acc:.4f}")
print(f"Validation AUC      : {auc:.4f}")
print(f"Validation Loss     : {loss:.4f}")


In [ ]:
# Cell 12: Classification report & confusion matrix

val_generator.reset()
y_pred_prob = model.predict(val_generator, verbose=1)
y_pred      = (y_pred_prob > 0.5).astype(int).flatten()
y_true      = val_generator.classes

# Map index -> name (e.g. {0: 'Fake', 1: 'Real'})
idx_to_class = {v: k for k, v in val_generator.class_indices.items()}
class_names  = [idx_to_class[i] for i in sorted(idx_to_class)]

print("\nClassification Report:")
print(classification_report(y_true, y_pred, target_names=class_names))

# Confusion matrix heatmap
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names)
plt.xlabel('Predicted'); plt.ylabel('Actual')
plt.title('Confusion Matrix')
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=150)
plt.show()


In [ ]:
# Cell 13: Save the final model
model.save('deepfake_detector_final.keras')
print("Model saved as deepfake_detector_final.keras")


In [ ]:
# Cell 14: (Optional) Predict on a single image
from tensorflow.keras.preprocessing import image as keras_image

def predict_image(img_path, model, threshold=0.5):
    img = keras_image.load_img(img_path, target_size=(224, 224))
    x   = keras_image.img_to_array(img)
    x   = preprocess_input(x)
    x   = np.expand_dims(x, axis=0)

    prob  = model.predict(x, verbose=0)[0][0]
    label = 'Real' if prob >= threshold else 'Fake'

    plt.imshow(img)
    plt.axis('off')
    plt.title(f"Prediction: {label}  (confidence: {prob:.2%})")
    plt.show()
    return label, prob

# Example usage:
# predict_image('/path/to/test_face.jpg', model)
print("predict_image() ready. Call it with any face crop.")
